# 🔬 TrustOCT-KD: Trustworthy Lightweight Retinal OCT Classification
## Calibration-Aware Knowledge Distillation with Explainability Preservation

---
### 📋 Notebook Sections
| # | Section | What It Does |
|---|---|---|
| 1 | Environment Setup | GPU check, install packages |
| 2 | Clone Repository | Clone from GitHub |
| 3 | Dataset Download | Kaggle API → Colab VM disk |
| 4 | Dataset Exploration | Visualize sample OCT images per class |
| 5 | Train Teacher | ResNet50+MSF+CBAM (epoch-by-epoch output) |
| 6 | Evaluate Teacher | Full metrics table |
| 7 | Train Student (No KD) | MobileNetV3 baseline |
| 8 | Knowledge Distillation | Teacher → Student with calibration-aware KD |
| 9 | Comparison Table | Side-by-side Teacher vs Student metrics |
| 10 | Calibration Analysis | ECE, Brier Score, Reliability Diagrams |
| 11 | Explainability | LayerCAM heatmaps comparison |
| 12 | AOPC Faithfulness | Quantitative deletion/insertion curves |
| 13 | Robustness | Noise, brightness, contrast perturbation |
| 14 | Model Complexity | Params, size, latency, FPS comparison |
| 15 | Save Results | Download ZIP or save to Drive |

---

---
## 📌 Section 1: Environment Setup

In [ ]:
!nvidia-smi

import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
!pip install -q kagglehub thop seaborn scikit-learn matplotlib tqdm opencv-python pandas
print("\n✅ All packages installed!")

---
## 📌 Section 2: Clone Repository

In [ ]:
import os

if not os.path.exists('TrustOCT-KD'):
    !git clone https://github.com/Gnanapravallika/TrustOCT-KD.git

%cd TrustOCT-KD
print("\n✅ Repository cloned! Files:")
!ls

---
## 📌 Section 3: Download Kermany OCT Dataset

📌 **Downloads to Colab VM disk (~5GB). No Google Drive needed!**

### Get your `kaggle.json`:
1. Go to [kaggle.com](https://www.kaggle.com) → Profile icon → **Settings**
2. Scroll to **API** → Click **Create New Token**
3. Upload the downloaded `kaggle.json` below

In [ ]:
# Upload your kaggle.json
from google.colab import files
uploaded = files.upload()

# Configure Kaggle API
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("\n✅ Kaggle API configured!")

In [ ]:
# Download dataset (~5GB, takes 3-10 minutes)
!kaggle datasets download -d paultimothymooney/kermany2018 -p data/ --unzip
print("\n✅ Dataset downloaded!")
!du -sh data/

In [ ]:
# Find and verify dataset structure
import os

DATA_DIR = 'data'
for root, dirs, f in os.walk(DATA_DIR):
    if 'train' in dirs and 'test' in dirs:
        DATA_DIR = root
        break

print(f"📁 Dataset root: {DATA_DIR}\n")
print(f"{'Split':<8} {'Class':<10} {'Count':>8}")
print("-" * 30)
for split in ['train', 'test', 'val']:
    split_dir = os.path.join(DATA_DIR, split)
    if os.path.exists(split_dir):
        for cls in sorted(os.listdir(split_dir)):
            cls_path = os.path.join(split_dir, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                print(f"{split:<8} {cls:<10} {count:>8}")

---
## 📌 Section 4: Dataset Exploration — Visualize OCT Samples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

classes = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, cls in enumerate(classes):
    cls_dir = os.path.join(DATA_DIR, 'train', cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpeg','.jpg','.png'))]
    
    for j in range(2):
        img = Image.open(os.path.join(cls_dir, imgs[j]))
        axes[j, i].imshow(np.array(img), cmap='gray')
        axes[j, i].set_title(f'{cls}', fontsize=14, fontweight='bold')
        axes[j, i].axis('off')

plt.suptitle('Sample Retinal OCT Images per Class', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("\n✅ Dataset visualization complete!")

---
## 📌 Section 5: Train Teacher Model (ResNet50 + MSF + CBAM)

⏱️ **Estimated time: ~1-2 hours on T4 GPU**

In [ ]:
import torch
import sys, os
sys.path.append('.')

from configs.config import Config
from trustoct.models import build_model
from trustoct.dataset.oct_dataset import get_dataloaders
from trustoct.training.trainer import Trainer, set_seed

Config.DATA_DIR = os.path.abspath(DATA_DIR)
Config.setup_directories()
set_seed(Config.SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️ Device: {device}")

# Create DataLoaders
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=Config.DATA_DIR,
    batch_size=Config.BATCH_SIZE,
    num_workers=2,
    image_size=Config.IMAGE_SIZE,
    use_clahe=True
)
print(f"📊 Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")

In [ ]:
# Build & Train Teacher Model
teacher_model = build_model('resnet50_msf_cbam', num_classes=4, pretrained=True)
t_params = sum(p.numel() for p in teacher_model.parameters()) / 1e6
print(f"\n🏗️ Teacher Model: ResNet50 + MSF + CBAM")
print(f"   Parameters: {t_params:.1f}M")

teacher_trainer = Trainer(
    model=teacher_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    lr=Config.LEARNING_RATE,
    num_epochs=Config.NUM_EPOCHS,
    checkpoint_dir=Config.CHECKPOINT_DIR,
    experiment_name='teacher_ResNet50_MSF_CBAM',
    use_amp=True
)

teacher_ckpt, teacher_history = teacher_trainer.fit()

In [ ]:
# Plot Teacher Training Curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(teacher_history['train_loss']) + 1)
ax1.plot(epochs, teacher_history['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax1.plot(epochs, teacher_history['val_loss'], 'r-s', label='Val Loss', markersize=4)
ax1.set_title('Teacher: Loss Curves', fontweight='bold', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, linestyle=':')

ax2.plot(epochs, [a*100 for a in teacher_history['train_acc']], 'b-o', label='Train Acc', markersize=4)
ax2.plot(epochs, [a*100 for a in teacher_history['val_acc']], 'r-s', label='Val Acc', markersize=4)
ax2.set_title('Teacher: Accuracy Curves', fontweight='bold', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, linestyle=':')

plt.tight_layout()
plt.savefig('outputs/visualizations/teacher_training_curves.png', dpi=300)
plt.show()
print(f"\n✅ Teacher training complete! Best Val Acc: {max(teacher_history['val_acc'])*100:.2f}%")

---
## 📌 Section 6: Evaluate Teacher Model

In [ ]:
from trustoct.evaluation.metrics import evaluate_classification, plot_confusion_matrix

# Load best teacher checkpoint
ckpt = torch.load(teacher_ckpt, map_location=device)
teacher_model.load_state_dict(ckpt['model_state_dict'])
teacher_model = teacher_model.to(device)

# Run evaluation
teacher_metrics, t_labels, t_preds, t_probs, t_cm = evaluate_classification(
    model=teacher_model, data_loader=test_loader, device=device
)

print("\n" + "="*50)
print(" 📊 TEACHER MODEL — Test Set Metrics")
print("="*50)
for k, v in teacher_metrics.items():
    print(f"  {k:<30} {v*100:.2f}%" if v <= 1 else f"  {k:<30} {v:.4f}")

# Confusion Matrix
plot_confusion_matrix(t_cm, classes, save_path='outputs/visualizations/teacher_confusion_matrix.png',
                      title='Teacher (ResNet50+MSF+CBAM) — Confusion Matrix')

from IPython.display import Image, display
display(Image(filename='outputs/visualizations/teacher_confusion_matrix.png', width=500))

---
## 📌 Section 7: Train Student WITHOUT Knowledge Distillation (Baseline)

⏱️ **Estimated time: ~30-60 min on T4 GPU**

In [ ]:
from trustoct.models import build_student

student_no_kd = build_student('mobilenetv3', num_classes=4, pretrained=True)
s_params = sum(p.numel() for p in student_no_kd.parameters()) / 1e6
print(f"\n🏗️ Student Model: MobileNetV3-Small")
print(f"   Parameters: {s_params:.1f}M (vs Teacher: {t_params:.1f}M)")
print(f"   Compression: {t_params/s_params:.1f}x smaller!")

student_no_kd_trainer = Trainer(
    model=student_no_kd,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    lr=Config.KD_LEARNING_RATE,
    num_epochs=Config.NUM_EPOCHS,
    checkpoint_dir=Config.CHECKPOINT_DIR,
    experiment_name='student_no_KD',
    use_amp=True
)

student_no_kd_ckpt, student_no_kd_history = student_no_kd_trainer.fit()
print(f"\n✅ Student (No KD) training complete!")

In [ ]:
# Evaluate Student (No KD)
ckpt = torch.load(student_no_kd_ckpt, map_location=device)
student_no_kd.load_state_dict(ckpt['model_state_dict'])
student_no_kd = student_no_kd.to(device)

student_no_kd_metrics, sn_labels, sn_preds, sn_probs, sn_cm = evaluate_classification(
    model=student_no_kd, data_loader=test_loader, device=device
)

print("\n" + "="*50)
print(" 📊 STUDENT (No KD) — Test Set Metrics")
print("="*50)
for k, v in student_no_kd_metrics.items():
    print(f"  {k:<30} {v*100:.2f}%" if v <= 1 else f"  {k:<30} {v:.4f}")

plot_confusion_matrix(sn_cm, classes, save_path='outputs/visualizations/student_no_kd_confusion_matrix.png',
                      title='Student (No KD) — Confusion Matrix')
display(Image(filename='outputs/visualizations/student_no_kd_confusion_matrix.png', width=500))

---
## 📌 Section 8: Knowledge Distillation (Teacher → Student)

🔥 **This is the core contribution of our paper!**

Loss = α·CE + β·KD + γ·Attention Transfer

⏱️ **Estimated time: ~1-1.5 hours on T4 GPU**

In [ ]:
from trustoct.training.distillation_trainer import DistillationTrainer

# Fresh student model for KD training
student_kd = build_student('mobilenetv3', num_classes=4, pretrained=True)

print(f"\n🔥 Starting Calibration-Aware Knowledge Distillation")
print(f"   Teacher: ResNet50+MSF+CBAM ({t_params:.1f}M params)")
print(f"   Student: MobileNetV3-Small ({s_params:.1f}M params)")
print(f"   Temperature: {Config.KD_TEMPERATURE}")
print(f"   Loss Weights: α={Config.KD_ALPHA} β={Config.KD_BETA} γ={Config.KD_GAMMA}")

distiller = DistillationTrainer(
    teacher_model=teacher_model,
    student_model=student_kd,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    lr=Config.KD_LEARNING_RATE,
    num_epochs=Config.KD_EPOCHS,
    temperature=Config.KD_TEMPERATURE,
    alpha=Config.KD_ALPHA,
    beta=Config.KD_BETA,
    gamma=Config.KD_GAMMA,
    checkpoint_dir=Config.CHECKPOINT_DIR,
    experiment_name='student_KD_TrustOCT',
    use_amp=True
)

student_kd_ckpt, kd_history = distiller.fit()
print(f"\n✅ Knowledge Distillation complete!")

In [ ]:
# Evaluate Student WITH KD
ckpt = torch.load(student_kd_ckpt, map_location=device)
student_kd.load_state_dict(ckpt['model_state_dict'])
student_kd = student_kd.to(device)

student_kd_metrics, sk_labels, sk_preds, sk_probs, sk_cm = evaluate_classification(
    model=student_kd, data_loader=test_loader, device=device
)

print("\n" + "="*50)
print(" 📊 STUDENT WITH KD (Ours) — Test Set Metrics")
print("="*50)
for k, v in student_kd_metrics.items():
    print(f"  {k:<30} {v*100:.2f}%" if v <= 1 else f"  {k:<30} {v:.4f}")

plot_confusion_matrix(sk_cm, classes, save_path='outputs/visualizations/student_kd_confusion_matrix.png',
                      title='Student WITH KD (Ours) — Confusion Matrix')
display(Image(filename='outputs/visualizations/student_kd_confusion_matrix.png', width=500))

---
## 📌 Section 9: Main Comparison Table (Paper Table 1)

In [ ]:
import pandas as pd

comparison_data = []
for name, metrics in [('Teacher (ResNet50+MSF+CBAM)', teacher_metrics),
                       ('Student w/o KD (MobileNetV3)', student_no_kd_metrics),
                       ('Student w/ KD (Ours)', student_kd_metrics)]:
    row = {'Model': name}
    for k, v in metrics.items():
        row[k] = f"{v*100:.2f}%" if v <= 1 else f"{v:.4f}"
    comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*70)
print(" 📊 PAPER TABLE 1: Classification Performance Comparison")
print("="*70)
display(df_comparison)

df_comparison.to_csv('outputs/results/classification_comparison.csv', index=False)
print("\n✅ Saved to outputs/results/classification_comparison.csv")

---
## 📌 Section 10: Calibration Analysis (ECE, Brier Score, Reliability Diagrams)

In [ ]:
from trustoct.evaluation.calibration import compute_calibration_metrics, plot_reliability_diagram

# Compute calibration for all 3 models
t_calib = compute_calibration_metrics(t_labels, t_probs)
sn_calib = compute_calibration_metrics(sn_labels, sn_probs)
sk_calib = compute_calibration_metrics(sk_labels, sk_probs)

print("\n" + "="*60)
print(" 📊 CALIBRATION ANALYSIS")
print("="*60)
print(f"{'Model':<35} {'ECE (%)':>10} {'Brier':>10}")
print("-" * 60)
print(f"{'Teacher (ResNet50+MSF+CBAM)':<35} {t_calib['ECE']*100:>9.2f}% {t_calib['Brier_Score']:>10.4f}")
print(f"{'Student w/o KD (MobileNetV3)':<35} {sn_calib['ECE']*100:>9.2f}% {sn_calib['Brier_Score']:>10.4f}")
print(f"{'Student w/ KD (Ours)':<35} {sk_calib['ECE']*100:>9.2f}% {sk_calib['Brier_Score']:>10.4f}")

if sk_calib['ECE'] < sn_calib['ECE']:
    improvement = ((sn_calib['ECE'] - sk_calib['ECE']) / sn_calib['ECE']) * 100
    print(f"\n🔥 KD improved calibration by {improvement:.1f}% (ECE reduction)!")

In [ ]:
# Reliability Diagrams side-by-side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, calib) in zip(axes, [
    ('Teacher', t_calib),
    ('Student w/o KD', sn_calib),
    ('Student w/ KD (Ours)', sk_calib)
]):
    num_bins = len(calib['bin_accs'])
    bin_centers = [0.05 + i * 0.1 for i in range(num_bins)]
    ax.bar(bin_centers, calib['bin_accs'], width=0.08, alpha=0.7, color='#2b5c8f', edgecolor='black')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2)
    ax.set_title(f'{name}\nECE: {calib["ECE"]*100:.2f}% | Brier: {calib["Brier_Score"]:.4f}',
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    ax.grid(True, linestyle=':')

plt.suptitle('Reliability Diagrams: Teacher vs Student Calibration', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('outputs/visualizations/reliability_diagrams_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Reliability diagrams saved!")

---
## 📌 Section 11: LayerCAM Explainability (Teacher vs Student Heatmaps)

In [ ]:
from trustoct.evaluation.explainability import LayerCAM, overlay_cam_on_image
import numpy as np

norm_mean = np.array([0.485, 0.456, 0.406]).reshape(1, 1, 3)
norm_std = np.array([0.229, 0.224, 0.225]).reshape(1, 1, 3)

# Setup LayerCAM for both models
teacher_cam = LayerCAM(teacher_model, target_layer=teacher_model.cbam_fused)
student_cam = LayerCAM(student_kd, target_layer=student_kd.features[-1])

# Get one sample per class
samples = {}
for images, labels in test_loader:
    for img, lbl in zip(images, labels):
        cls_idx = lbl.item()
        if cls_idx not in samples:
            samples[cls_idx] = img
        if len(samples) >= 4:
            break
    if len(samples) >= 4:
        break

fig, axes = plt.subplots(4, 3, figsize=(14, 18))

for row, (cls_idx, img_tensor) in enumerate(sorted(samples.items())):
    cls_name = classes[cls_idx]
    img_input = img_tensor.unsqueeze(0).to(device)

    img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img_np = np.clip(img_np * norm_std + norm_mean, 0, 1)

    t_map, t_pred, t_prob = teacher_cam.generate(img_input, target_class=cls_idx)
    s_map, s_pred, s_prob = student_cam.generate(img_input, target_class=cls_idx)

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f'Original ({cls_name})', fontweight='bold', fontsize=12)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(overlay_cam_on_image(img_np, t_map))
    axes[row, 1].set_title(f'Teacher CAM ({t_prob*100:.1f}%)', fontweight='bold', fontsize=12)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(overlay_cam_on_image(img_np, s_map))
    axes[row, 2].set_title(f'Student CAM ({s_prob*100:.1f}%)', fontweight='bold', fontsize=12)
    axes[row, 2].axis('off')

plt.suptitle('LayerCAM: Teacher vs Student Explainability', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/visualizations/layercam_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ LayerCAM heatmaps saved!")

---
## 📌 Section 12: AOPC Faithfulness (Quantitative Explanation Verification)

In [ ]:
from trustoct.evaluation.explainability import compute_aopc_faithfulness

print("\n" + "="*60)
print(" 📊 AOPC FAITHFULNESS COMPARISON")
print("="*60)
print(f"{'Class':<10} {'Teacher Del-AOPC':>18} {'Student Del-AOPC':>18} {'Teacher Ins-AOPC':>18} {'Student Ins-AOPC':>18}")
print("-" * 85)

aopc_rows = []
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (cls_idx, img_tensor) in enumerate(sorted(samples.items())):
    cls_name = classes[cls_idx]
    img_input = img_tensor.unsqueeze(0).to(device)

    t_map, _, _ = teacher_cam.generate(img_input, target_class=cls_idx)
    s_map, _, _ = student_cam.generate(img_input, target_class=cls_idx)

    t_aopc = compute_aopc_faithfulness(teacher_model, img_input, t_map, cls_idx, device=device)
    s_aopc = compute_aopc_faithfulness(student_kd, img_input, s_map, cls_idx, device=device)

    print(f"{cls_name:<10} {t_aopc['deletion_aopc']:>18.4f} {s_aopc['deletion_aopc']:>18.4f} {t_aopc['insertion_aopc']:>18.4f} {s_aopc['insertion_aopc']:>18.4f}")

    aopc_rows.append({'Class': cls_name, 'Teacher Del-AOPC': t_aopc['deletion_aopc'],
                      'Student Del-AOPC': s_aopc['deletion_aopc'],
                      'Teacher Ins-AOPC': t_aopc['insertion_aopc'],
                      'Student Ins-AOPC': s_aopc['insertion_aopc']})

    ax = axes[idx // 2, idx % 2]
    ax.plot(t_aopc['percentages'], t_aopc['deletion_scores'], 'r-o', label=f'Teacher (AOPC={t_aopc["deletion_aopc"]:.3f})', markersize=4)
    ax.plot(s_aopc['percentages'], s_aopc['deletion_scores'], 'b-s', label=f'Student (AOPC={s_aopc["deletion_aopc"]:.3f})', markersize=4)
    ax.set_title(f'{cls_name} — Deletion Curve', fontweight='bold')
    ax.set_xlabel('% Pixels Deleted'); ax.set_ylabel('Confidence')
    ax.legend(fontsize=9); ax.grid(True, linestyle=':')

plt.suptitle('AOPC Faithfulness: Teacher vs Student', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/visualizations/aopc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

aopc_df = pd.DataFrame(aopc_rows)
aopc_df.to_csv('outputs/results/aopc_comparison.csv', index=False)
print("\n✅ AOPC comparison saved!")

---
## 📌 Section 13: Robustness Under Clinical Perturbations

In [ ]:
from trustoct.evaluation.robustness import evaluate_robustness

print("\n⏳ Evaluating robustness (Teacher)...")
t_robust = evaluate_robustness(teacher_model, test_loader, device=device)

print("⏳ Evaluating robustness (Student w/o KD)...")
sn_robust = evaluate_robustness(student_no_kd, test_loader, device=device)

print("⏳ Evaluating robustness (Student w/ KD)...")
sk_robust = evaluate_robustness(student_kd, test_loader, device=device)

print("\n" + "="*80)
print(" 📊 ROBUSTNESS COMPARISON")
print("="*80)
print(f"{'Perturbation':<30} {'Teacher':>12} {'No KD':>12} {'KD (Ours)':>12}")
print("-" * 70)

for perturbation in t_robust.keys():
    t_acc = t_robust[perturbation]['Accuracy'] * 100
    sn_acc = sn_robust[perturbation]['Accuracy'] * 100
    sk_acc = sk_robust[perturbation]['Accuracy'] * 100
    print(f"{perturbation:<30} {t_acc:>11.2f}% {sn_acc:>11.2f}% {sk_acc:>11.2f}%")

print("\n✅ Robustness evaluation complete!")

---
## 📌 Section 14: Model Complexity Comparison

In [ ]:
from trustoct.evaluation.benchmark import profile_model_complexity

print("⏳ Profiling Teacher...")
t_complexity = profile_model_complexity(teacher_model, device=device)

print("⏳ Profiling Student (KD)...")
sk_complexity = profile_model_complexity(student_kd, device=device)

print("\n" + "="*60)
print(" 📊 MODEL COMPLEXITY COMPARISON")
print("="*60)
print(f"{'Metric':<25} {'Teacher':>18} {'Student (Ours)':>18}")
print("-" * 60)
for key in t_complexity:
    print(f"{key:<25} {t_complexity[key]:>18} {sk_complexity[key]:>18}")

print("\n✅ Complexity comparison complete!")

---
## 📌 Section 15: Save All Results

In [ ]:
# Option A: Download as ZIP to your computer
!zip -r TrustOCT_Results.zip outputs/results/ outputs/visualizations/

from google.colab import files
files.download('TrustOCT_Results.zip')
print("\n✅ Results downloaded to your computer!")

In [ ]:
# Option B: Save checkpoints + results to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
save_dir = '/content/drive/MyDrive/TrustOCT_Results'
os.makedirs(save_dir, exist_ok=True)

for folder in ['results', 'visualizations']:
    src = f'outputs/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, f'{save_dir}/{folder}', dirs_exist_ok=True)

for ckpt in glob.glob('outputs/checkpoints/*_best.pth'):
    shutil.copy(ckpt, save_dir)

print(f"\n✅ Everything saved to Google Drive: {save_dir}")

---
## 📋 Paper Artifacts Checklist

| # | Artifact | File | ✅ |
|---|---|---|---|
| 1 | Classification metrics table | `outputs/results/classification_comparison.csv` | ☐ |
| 2 | AOPC faithfulness table | `outputs/results/aopc_comparison.csv` | ☐ |
| 3 | Teacher training curves | `outputs/visualizations/teacher_training_curves.png` | ☐ |
| 4 | Confusion matrices (×3) | `outputs/visualizations/*_confusion_matrix.png` | ☐ |
| 5 | Reliability diagrams | `outputs/visualizations/reliability_diagrams_comparison.png` | ☐ |
| 6 | LayerCAM heatmaps | `outputs/visualizations/layercam_comparison.png` | ☐ |
| 7 | AOPC deletion curves | `outputs/visualizations/aopc_comparison.png` | ☐ |
| 8 | Model checkpoints | `outputs/checkpoints/*_best.pth` | ☐ |